# Wan 2.2 (TI2V-5B) — Video Clip Generation (Colab T4)

Generates a short clip per storyboard shot, conditioned on the Cloudflare
keyframes packaged into the job's `inputs/` folder.

Run **top to bottom**. All real work lives in `gpu_video.py` (synced into the
exchange folder by `python -m pipeline.run --stage package`).

**T4 notes**: fp16 + model CPU offload + VAE tiling keep VRAM under ~14.5 GB.
Shots are deliberately small (480x480, ~2s). If a shot OOMs or fails, the
local assemble stage falls back to an ffmpeg zoompan clip from the keyframe —
the film still completes.

In [ ]:
# 1. Verify the GPU (Runtime > Change runtime type > T4 GPU)
import torch
assert torch.cuda.is_available(), 'No GPU — enable T4: Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Mount Google Drive and point at the exchange folder
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
EXCHANGE = Path('/content/drive/MyDrive/heritage-exchange')
assert (EXCHANGE / 'pipeline_gpu').is_dir(), (
    'Exchange folder not found. Run `python -m pipeline.run --stage package` '
    'locally, then sync/upload the exchange dir to My Drive as heritage-exchange.'
)

In [ ]:
# 3. Install dependencies (~5-8 min: diffusers + transformers + video export)
!pip install -q diffusers transformers accelerate sentencepiece ftfy 'imageio[ffmpeg]'

In [ ]:
# 4. Import the synced generation module
import sys
sys.path.insert(0, str(EXCHANGE / 'pipeline_gpu'))
import gpu_video

In [ ]:
# 5. Smoke test: a tiny 17-frame clip (~few min incl. ~10GB model download)
smoke = gpu_video.run_smoke(EXCHANGE)
print(smoke['status'], smoke.get('files') or smoke.get('error'))

In [ ]:
# 6. Run the packaged video job — one clip per storyboard shot
jobs = sorted((EXCHANGE / 'jobs').glob('video-*/job.json'))
assert jobs, 'No video job found — run `python -m pipeline.run --stage package` locally first.'
entry = gpu_video.run_job(jobs[-1], EXCHANGE)
print(entry['status'], entry.get('files') or entry.get('error'))

## Done

Results are in `heritage-exchange/results/`. Wait for Drive to sync them to
your machine (or download the folder manually), then run locally:

```bash
python -m pipeline.run --resume
```

Shots that failed on GPU are covered by the ffmpeg keyframe fallback during
assemble — check `runs.jsonl` for `status: fallback` entries.